# F3：从经历中学出第一台小世界

这一次拿走人工转移表。我们在带打滑的 LineWorld 中收集 episode，用计数学习概率 dynamics，再把它交给 MPC。

In [ ]:
from pathlib import Path
import random
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from hwm.gridworld import EmpiricalDynamics, LineWorld, mpc_episode
print('环境检查通过。')

## 1. 收集连续 episode

一维世界左端是陷阱，右端是终点。动作有 left 与 right，20% 概率打滑并停在原地。

In [ ]:
world = LineWorld(slip_probability=0.2)
random_policy = lambda state, rng: rng.choice(world.actions)
transitions, episode_ids = world.collect(
    random_policy, episodes=200, max_steps=20, seed=4
)
print('第一帧:', world.render(world.start))
print('transition 数:', len(transitions))
print('episode 数:', len(set(episode_ids)))
for item in transitions[:4]:
    print(item)
assert len(transitions) == len(episode_ids)

每条记录都有当前状态、动作、奖励、下一状态与 done。`episode_ids` 让我们知道哪些相邻记录属于同一段经历。

## 2. 按 episode 切分，不随机拆 transition

In [ ]:
train_ids = set(range(0, 140))
val_ids = set(range(140, 170))
test_ids = set(range(170, 200))
train = [t for t, i in zip(transitions, episode_ids) if i in train_ids]
val = [t for t, i in zip(transitions, episode_ids) if i in val_ids]
test = [t for t, i in zip(transitions, episode_ids) if i in test_ids]
print('train/val/test:', len(train), len(val), len(test))
print('episode id 是否重叠:', bool(
    train_ids & val_ids or train_ids & test_ids or val_ids & test_ids
))
assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)

## 3. 用计数学习多种下一状态

模型不读取真实移动规则，只统计训练数据中 `(state, action)` 后面出现过什么。

In [ ]:
model = EmpiricalDynamics().fit(train)
distribution = model.distribution(world.start, 'right')
print(f'P(next_state | state={world.start}, action=right)')
for state, probability in distribution.items():
    print(f'  {state}: {probability:.3f}')
print('真实设定：停留概率 0.2，向右概率 0.8')
assert world.start in distribution and world.start + 1 in distribution

有限数据不会恰好得到 0.2 与 0.8。样本增加后，经验频率通常会靠近环境概率。

## 4. 一步准确与多步成功不是同一指标

In [ ]:
correct = 0
known = 0
for item in test:
    predicted = model.transition(item.state, item.action)
    if model.distribution(item.state, item.action):
        known += 1
        correct += predicted.next_state == item.next_state
print('已见状态上的一步准确率:', round(correct / known, 3))
print('测试 transition 覆盖率:', round(known / len(test), 3))
assert known > 0

打滑本来就带来随机性，所以最可能结果的一步准确率不会达到 1。更重要的问题是：使用这台模型以后，任务能否完成。

## 5. 固定同一起点，只替换动作

In [ ]:
for action in world.actions:
    print(action, '->', model.distribution(world.start, action))
assert model.distribution(world.start, 'left') != model.distribution(
    world.start, 'right'
)

历史、状态和环境参数相同，只有动作改变。两个分布不同，说明模型确实读取了动作。

## 6. 在模型里规划，在环境里只走一步

In [ ]:
steps, plans = mpc_episode(
    world, model, depth=4, max_steps=20, seed=7,
    action_order=world.actions,
)
for index, (step, plan) in enumerate(zip(steps, plans), start=1):
    print(
        f'{index:>2}. {step.state} --{plan.action}--> '
        f'{step.next_state}, reward={step.reward}'
    )
print('最后状态:', steps[-1].next_state, '总回报:', sum(x.reward for x in steps))
assert steps[-1].next_state == world.goal

这就是第一台从经历学出的世界模型闭环。它很小，但已经包含真实数据、概率动态、反事实、规划和现实修正。

## 7. 数据少时会发生什么

只用前 2 个 episode 训练，再检查模型覆盖了多少测试 transition。

In [ ]:
tiny_train = [t for t, i in zip(transitions, episode_ids) if i < 2]
tiny_model = EmpiricalDynamics().fit(tiny_train)
tiny_known = sum(bool(tiny_model.distribution(t.state, t.action)) for t in test)
full_known = sum(bool(model.distribution(t.state, t.action)) for t in test)
print('2 个 episode 覆盖:', round(tiny_known / len(test), 3))
print('140 个 episode 覆盖:', round(full_known / len(test), 3))
assert tiny_known <= full_known

## 小结与 PA0 入口

- [ ] 我能从 episode 构造对齐的 transition。
- [ ] 我知道为什么 split 不能随机拆相邻步骤。
- [ ] 我能从计数得到下一状态分布。
- [ ] 我能同时报告一步准确率、覆盖率和闭环结果。
- [ ] 我能用反事实检查模型是否读取动作。

PA0 会换一张地图，并拿走若干实现。不要先添加神经网络。先找出表格模型在哪个稳定条件下失败，再决定下一条路线需要什么。